# Linguagens de Programação para Engenharia de Dados
## Encontro 1 — Notebook Prático

**Instituição:** Universidade de Fortaleza (UNIFOR)
**Curso:** Pós-Graduação — Especialização em Engenharia de Dados
**Professor:** Cassio Pinheiro — [LinkedIn](https://www.linkedin.com/in/cassio-pinheiro-9baa7939/)
**Data:** 25/06/2026 · 19:00 às 22:30

---

### Como usar este notebook

1. Execute as células **na ordem**, de cima para baixo (`Shift + Enter`).
2. Cada seção corresponde a uma seção do documento `.md` do encontro.
3. O Encontro 1 roda **100% com a biblioteca padrão do Python**. Há um único ponto opcional (seção 3) que usa `duckdb` — se não estiver instalado, o notebook segue normalmente. `pandas` entra a partir do Encontro 2.
4. **Mexa no código.** Troque valores, quebre de propósito, conserte. É assim que se aprende.

> **Projeto fio condutor:** um arquivo `vendas.csv` sintético e *deliberadamente sujo*. Vamos levá-lo do bruto até um dado confiável e, ao final, montar nosso primeiro mini-pipeline ETL.


## Setup — só o necessário

Apenas a **biblioteca padrão** do Python. A seção 3 tem um trecho **opcional** com `duckdb` (SQL no notebook). `pandas` só entra a partir do Encontro 2.

In [1]:
import csv
import io
from datetime import datetime

print("Setup OK — usando apenas a biblioteca padrão do Python por enquanto.")


Setup OK — usando apenas a biblioteca padrão do Python por enquanto.


---

# 1 — Mapa do território

> 📖 **No `.md`:** seção *1 — O que é Engenharia de Dados*.

Antes de qualquer código, vamos desenhar o ciclo de vida da Engenharia de Dados.
Não é um diagrama bonito por enfeite: é o mapa que organiza **tudo** que faremos no curso.


In [2]:
# O ciclo de vida da Engenharia de Dados, descrito em dados (não em PowerPoint!)
ciclo = [
    ("1. Geração",      "O dado nasce: um sistema, um sensor, um formulário, uma API"),
    ("2. Ingestão",     "Trazemos o dado para o nosso ambiente"),
    ("3. Armazenamento","Guardamos de forma durável e organizada"),
    ("4. Transformação","Limpamos, validamos, enriquecemos, modelamos"),
    ("5. Entrega",      "Servimos para análise, IA, dashboards, negócio"),
]

print("CICLO DE VIDA DA ENGENHARIA DE DADOS\n" + "=" * 55)
for etapa, descricao in ciclo:
    print(f"{etapa:18s} -> {descricao}")

print("\nObserve: a Ciência de Dados consome o RESULTADO da etapa 5.")
print("Sem as etapas 1-5 funcionando, não há o que analisar.")


CICLO DE VIDA DA ENGENHARIA DE DADOS
1. Geração         -> O dado nasce: um sistema, um sensor, um formulário, uma API
2. Ingestão        -> Trazemos o dado para o nosso ambiente
3. Armazenamento   -> Guardamos de forma durável e organizada
4. Transformação   -> Limpamos, validamos, enriquecemos, modelamos
5. Entrega         -> Servimos para análise, IA, dashboards, negócio

Observe: a Ciência de Dados consome o RESULTADO da etapa 5.
Sem as etapas 1-5 funcionando, não há o que analisar.


**Engenheiro de dados x Cientista de dados — a diferença em uma linha:**

> Se o dado é água: o *engenheiro* constrói o encanamento e o tratamento; o *cientista* analisa a água para descobrir o que ela revela. **Sem encanamento, não há análise.**


---

# 2 — Do bruto ao confiável

> 📖 **No `.md`:** seção *2 — O dado como produto*.

Vamos **criar de propósito** um arquivo de vendas com os problemas que aparecem no mundo real:
datas em formatos diferentes, valores faltando, valor negativo, duplicata, espaços sobrando.
Esse é o nosso "dado bruto" — a água não tratada.


In [3]:
# CSV sintético SUJO de propósito — representa o que chega de um sistema legado.
# (texto multi-linha simulando o conteúdo de um arquivo vendas.csv)
csv_bruto = """id,data,cliente,categoria,valor
1,2026-06-01,Ana Souza,eletronicos,1200.00
2,01/06/2026,  Bruno Lima ,alimentacao,85.50
3,2026-06-02,Carla Dias,eletronicos,
4,2026-06-02,Diego Reis,transporte,-30.00
5,2026/06/03,Ana Souza,alimentacao,42.90
5,2026/06/03,Ana Souza,alimentacao,42.90
6,2026-06-03,Elena Cruz,vestuario,260.00
7,,Felipe Aragao,servicos,99.90
"""

print(csv_bruto)
print("Repare nos problemas: data em 3 formatos, valor vazio (id 3),")
print("valor negativo (id 4), linha duplicada (id 5), data vazia (id 7),")
print("e espaços sobrando no cliente (id 2).")


id,data,cliente,categoria,valor
1,2026-06-01,Ana Souza,eletronicos,1200.00
2,01/06/2026,  Bruno Lima ,alimentacao,85.50
3,2026-06-02,Carla Dias,eletronicos,
4,2026-06-02,Diego Reis,transporte,-30.00
5,2026/06/03,Ana Souza,alimentacao,42.90
5,2026/06/03,Ana Souza,alimentacao,42.90
6,2026-06-03,Elena Cruz,vestuario,260.00
7,,Felipe Aragao,servicos,99.90

Repare nos problemas: data em 3 formatos, valor vazio (id 3),
valor negativo (id 4), linha duplicada (id 5), data vazia (id 7),
e espaços sobrando no cliente (id 2).


Vamos ler esse texto como se fosse um arquivo, usando o módulo `csv` da biblioteca padrão:

In [4]:
# Ler o CSV (de uma string, via io.StringIO — simula abrir um arquivo)
leitor = csv.DictReader(io.StringIO(csv_bruto))
linhas_brutas = list(leitor)

print(f"Linhas lidas: {len(linhas_brutas)}\n")
for linha in linhas_brutas:
    print(linha)

# Cada linha virou um DICIONÁRIO {coluna: valor}. Guarde essa ideia:
# uma tabela, em Python, é naturalmente uma LISTA DE DICIONÁRIOS.


Linhas lidas: 8

{'id': '1', 'data': '2026-06-01', 'cliente': 'Ana Souza', 'categoria': 'eletronicos', 'valor': '1200.00'}
{'id': '2', 'data': '01/06/2026', 'cliente': '  Bruno Lima ', 'categoria': 'alimentacao', 'valor': '85.50'}
{'id': '3', 'data': '2026-06-02', 'cliente': 'Carla Dias', 'categoria': 'eletronicos', 'valor': ''}
{'id': '4', 'data': '2026-06-02', 'cliente': 'Diego Reis', 'categoria': 'transporte', 'valor': '-30.00'}
{'id': '5', 'data': '2026/06/03', 'cliente': 'Ana Souza', 'categoria': 'alimentacao', 'valor': '42.90'}
{'id': '5', 'data': '2026/06/03', 'cliente': 'Ana Souza', 'categoria': 'alimentacao', 'valor': '42.90'}
{'id': '6', 'data': '2026-06-03', 'cliente': 'Elena Cruz', 'categoria': 'vestuario', 'valor': '260.00'}
{'id': '7', 'data': '', 'cliente': 'Felipe Aragao', 'categoria': 'servicos', 'valor': '99.90'}


---

# 3 — Por que código (e não cliques)

> 📖 **No `.md`:** seção *3 — Por que programar*.

A mesma pergunta — *"qual o faturamento por categoria?"* — respondida de três formas.
Compare o esforço de fazer "na mão" com o de automatizar.


In [5]:
# (a) "Na mão" — imagine somar isso numa calculadora, todo dia, sem errar.
#     Com 8 linhas já é chato. Com 8 milhões, é impossível.

# (b) Automatizado em Python: escreve uma vez, roda sempre, sobre qualquer tamanho.
faturamento = {}
for linha in linhas_brutas:
    cat = linha["categoria"]
    valor_txt = linha["valor"]
    if valor_txt.strip() == "":      # pula valores vazios por enquanto
        continue
    valor = float(valor_txt)
    faturamento[cat] = faturamento.get(cat, 0.0) + valor

print("Faturamento por categoria (Python puro):")
for cat, total in faturamento.items():
    print(f"  {cat:14s} R$ {total:>9.2f}")

print("\nNote: o resultado ainda está ERRADO (inclui o valor negativo do id 4).")
print("É por isso que LIMPAR vem antes de ANALISAR. Faremos isso na seção 6.")


Faturamento por categoria (Python puro):
  eletronicos    R$   1200.00
  alimentacao    R$    171.30
  transporte     R$    -30.00
  vestuario      R$    260.00
  servicos       R$     99.90

Note: o resultado ainda está ERRADO (inclui o valor negativo do id 4).
É por isso que LIMPAR vem antes de ANALISAR. Faremos isso na seção 6.


**A mesma pergunta em SQL** — a outra língua que você usará a vida toda em dados.
Usamos o DuckDB, um banco que roda dentro do próprio notebook (se não estiver instalado, a célula apenas mostra o SQL):

In [6]:
consulta_sql = """
SELECT categoria, SUM(CAST(valor AS DOUBLE)) AS faturamento
FROM vendas
WHERE valor IS NOT NULL AND valor != \'\'
GROUP BY categoria
ORDER BY faturamento DESC
"""

try:
    import duckdb
    con = duckdb.connect()
    con.execute("""CREATE TABLE vendas (id INT, data VARCHAR, cliente VARCHAR,
                                        categoria VARCHAR, valor VARCHAR)""")
    for l in linhas_brutas:
        con.execute("INSERT INTO vendas VALUES (?,?,?,?,?)",
                    [l["id"], l["data"], l["cliente"], l["categoria"], l["valor"]])
    print("Resultado via SQL (DuckDB):")
    print(con.execute(consulta_sql).fetchdf())
except ImportError:
    print("DuckDB não instalado — sem problemas. O SQL que faria o mesmo trabalho:")
    print(consulta_sql)
    print(">> Repare: SQL descreve O QUE QUEREMOS, não o passo a passo. Essa é a")
    print(">> diferença de pensar em CONJUNTOS (SQL) vs. em PASSOS (Python).")


Resultado via SQL (DuckDB):
     categoria  faturamento
0  eletronicos       1200.0
1    vestuario        260.0
2  alimentacao        171.3
3     servicos         99.9
4   transporte        -30.0


---

# 4 — Os blocos do Python

> 📖 **No `.md`:** seção *4 — Python como linguagem de cola*.

Cada bloco abaixo é uma **peça de pipeline**. Execute, leia o resultado, depois altere os valores.


In [7]:
# VARIÁVEIS e TIPOS
texto    = "transacao"   # str
inteiro  = 42            # int
decimal  = 19.90         # float
booleano = True          # bool
vazio    = None          # ausência de valor — onipresente em dados reais!

for valor in [texto, inteiro, decimal, booleano, vazio]:
    print(f"{str(valor):12s} -> tipo: {type(valor).__name__}")


transacao    -> tipo: str
42           -> tipo: int
19.9         -> tipo: float
True         -> tipo: bool
None         -> tipo: NoneType


In [8]:
# OPERAÇÕES e CONDICIONAIS — uma regra de negócio simples
taxa_imposto = 0.18

def classificar(subtotal):
    total = subtotal * (1 + taxa_imposto)   # operação
    if total > 1000:                        # condicional
        faixa = "alto valor"
    elif total > 100:
        faixa = "valor medio"
    else:
        faixa = "baixo valor"
    return total, faixa

for s in [50.0, 250.0, 1200.0]:
    total, faixa = classificar(s)
    print(f"subtotal R$ {s:>8.2f}  ->  total R$ {total:>8.2f}  [{faixa}]")


subtotal R$    50.00  ->  total R$    59.00  [baixo valor]
subtotal R$   250.00  ->  total R$   295.00  [valor medio]
subtotal R$  1200.00  ->  total R$  1416.00  [alto valor]


In [9]:
# LAÇO (loop) — o coração de qualquer pipeline: repetir sobre muitos itens
valores = [10.0, 250.0, 75.5, 5.0]

soma = 0.0
for v in valores:
    soma = soma + v

print(f"Soma de {len(valores)} valores: R$ {soma:.2f}")
print(f"Média: R$ {soma / len(valores):.2f}")

# Experimente: adicione mais valores à lista e rode de novo.
# O código NÃO MUDA — funciona para 4 ou 4 milhões de itens.


Soma de 4 valores: R$ 340.50
Média: R$ 85.12


---

# 5 — Estruturas de dados

> 📖 **No `.md`:** seção *5 — Tipos e estruturas*.

Lista, dicionário, tupla e conjunto: as quatro peças com que se monta qualquer fluxo de dados.


In [10]:
# LISTA — sequência ordenada e mutável
arquivos = ["jan.csv", "fev.csv", "mar.csv"]
arquivos.append("abr.csv")
print("Lista:", arquivos)
print("Primeiro item (índice 0):", arquivos[0], "| Total:", len(arquivos))

# DICIONÁRIO — pares chave:valor = um REGISTRO (uma linha de tabela)
transacao = {"id": 1001, "valor": 250.00, "categoria": "eletronicos", "aprovada": True}
transacao["device"] = "mobile"      # adiciona campo
print("\nDicionário (1 registro):", transacao)
print("Acesso por chave -> valor:", transacao["valor"])

# TUPLA — sequência IMUTÁVEL (não muda depois de criada)
fortaleza = (-3.7327, -38.5267)
print("\nTupla (coordenada fixa):", fortaleza)

# CONJUNTO (set) — sem ordem, SEM DUPLICATAS
categorias = {"alimentacao", "transporte", "alimentacao", "eletronicos"}
print("\nConjunto (duplicata sumiu):", categorias)
print("'transporte' está presente?", "transporte" in categorias)


Lista: ['jan.csv', 'fev.csv', 'mar.csv', 'abr.csv']
Primeiro item (índice 0): jan.csv | Total: 4

Dicionário (1 registro): {'id': 1001, 'valor': 250.0, 'categoria': 'eletronicos', 'aprovada': True, 'device': 'mobile'}
Acesso por chave -> valor: 250.0

Tupla (coordenada fixa): (-3.7327, -38.5267)

Conjunto (duplicata sumiu): {'alimentacao', 'eletronicos', 'transporte'}
'transporte' está presente? True


In [11]:
# A ponte para o pandas: uma LISTA DE DICIONÁRIOS já é, na prática, uma TABELA.
tabela = [
    {"id": 1, "categoria": "eletronicos", "valor": 1200.0},
    {"id": 2, "categoria": "alimentacao", "valor":   85.5},
    {"id": 3, "categoria": "eletronicos", "valor":  260.0},
]

# Valores únicos de categoria? Um set resolve em uma linha:
categorias_unicas = {linha["categoria"] for linha in tabela}
print("Categorias distintas:", categorias_unicas)

# Faturamento por categoria? Um dicionário acumula:
fat = {}
for linha in tabela:
    fat[linha["categoria"]] = fat.get(linha["categoria"], 0.0) + linha["valor"]
print("Faturamento:", fat)

print("\nNas próximas aulas, o pandas fará isso com UMA linha. Mas a IDEIA")
print("é exatamente esta: colunas nomeadas (dict) sobre linhas ordenadas (list).")


Categorias distintas: {'alimentacao', 'eletronicos'}
Faturamento: {'eletronicos': 1460.0, 'alimentacao': 85.5}

Nas próximas aulas, o pandas fará isso com UMA linha. Mas a IDEIA
é exatamente esta: colunas nomeadas (dict) sobre linhas ordenadas (list).


---

# 6 — Seu primeiro pipeline (ETL)

> 📖 **No `.md`:** seção *6 — Primeiro mini-pipeline*.

Agora juntamos tudo. Vamos pegar o `vendas.csv` **bruto** da seção 2 e levá-lo até **dado confiável**, no padrão **Extrair → Transformar → Carregar** — contando o que mantivemos e o que descartamos (rastreabilidade!).


In [12]:
# ---------- EXTRAIR ----------
linhas = list(csv.DictReader(io.StringIO(csv_bruto)))
print(f"[EXTRAIR]  {len(linhas)} linhas brutas lidas.")

# ---------- TRANSFORMAR ----------
def padronizar_data(texto):
    """Aceita AAAA-MM-DD, DD/MM/AAAA ou AAAA/MM/DD; devolve AAAA-MM-DD ou None."""
    texto = (texto or "").strip()
    for formato in ("%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d"):
        try:
            return datetime.strptime(texto, formato).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None

limpas = []
descartes = {"valor_ausente": 0, "valor_negativo": 0, "data_invalida": 0, "duplicata": 0}
vistos = set()                          # para deduplicação

for linha in linhas:
    # regra 1: valor precisa existir
    if (linha["valor"] or "").strip() == "":
        descartes["valor_ausente"] += 1
        continue
    valor = float(linha["valor"])
    # regra 2: valor não pode ser negativo
    if valor < 0:
        descartes["valor_negativo"] += 1
        continue
    # regra 3: data precisa ser válida
    data = padronizar_data(linha["data"])
    if data is None:
        descartes["data_invalida"] += 1
        continue
    # regra 4: sem duplicatas (chave = id + data + valor)
    chave = (linha["id"], data, valor)
    if chave in vistos:
        descartes["duplicata"] += 1
        continue
    vistos.add(chave)

    # enriquecimento: padroniza texto e calcula total com imposto
    limpas.append({
        "id": int(linha["id"]),
        "data": data,
        "cliente": linha["cliente"].strip(),
        "categoria": linha["categoria"].strip(),
        "valor": valor,
        "total_com_imposto": round(valor * 1.18, 2),
    })

print(f"[TRANSFORMAR]  {len(limpas)} linhas mantidas.")
print(f"               descartes: {descartes}")

# ---------- CARREGAR ----------
buffer = io.StringIO()
campos = ["id", "data", "cliente", "categoria", "valor", "total_com_imposto"]
escritor = csv.DictWriter(buffer, fieldnames=campos)
escritor.writeheader()
escritor.writerows(limpas)

print("\n[CARREGAR]  vendas_tratadas.csv gerado:\n")
print(buffer.getvalue())


[EXTRAIR]  8 linhas brutas lidas.
[TRANSFORMAR]  4 linhas mantidas.
               descartes: {'valor_ausente': 1, 'valor_negativo': 1, 'data_invalida': 1, 'duplicata': 1}

[CARREGAR]  vendas_tratadas.csv gerado:

id,data,cliente,categoria,valor,total_com_imposto
1,2026-06-01,Ana Souza,eletronicos,1200.0,1416.0
2,2026-06-01,Bruno Lima,alimentacao,85.5,100.89
5,2026-06-03,Ana Souza,alimentacao,42.9,50.62
6,2026-06-03,Elena Cruz,vestuario,260.0,306.8



### O que tornou isto *engenharia*, e não só um script

- **Rastreabilidade:** contamos exatamente quantas linhas entraram, saíram e *por que* cada uma foi descartada.
- **Idempotência:** rodar de novo dá o mesmo resultado — a deduplicação garante isso.
- **Tratamento de erro:** datas inválidas e valores ausentes **não quebram** o pipeline; são tratados explicitamente.
- **Contrato de saída:** a tabela final tem colunas e tipos previsíveis, prontos para o próximo consumidor.

Nas próximas aulas, **pandas** substitui esses laços por operações de tabela, **SQL** empurra trabalho para o banco, e a **orquestração** faz tudo rodar sozinho na hora certa. O esqueleto **E→T→L** permanece.

In [13]:
# Confirmação final: o faturamento por categoria AGORA está correto
# (sem o valor negativo e sem a duplicata que poluíam a seção 3).
fat_correto = {}
for l in limpas:
    fat_correto[l["categoria"]] = fat_correto.get(l["categoria"], 0.0) + l["valor"]

print("Faturamento por categoria (dado JÁ tratado):")
for cat, total in sorted(fat_correto.items(), key=lambda x: -x[1]):
    print(f"  {cat:14s} R$ {total:>9.2f}")


Faturamento por categoria (dado JÁ tratado):
  eletronicos    R$   1200.00
  vestuario      R$    260.00
  alimentacao    R$    128.40


---

## 🧪 Exercícios do Encontro 1

Resolva no próprio notebook, criando novas células abaixo:

1. **Aquecimento.** Na seção 4, crie uma função `aplicar_desconto(valor, percentual)` que devolve o valor com desconto. Teste com 3 valores.
2. **Estruturas.** Na seção 5, a partir da lista `tabela`, descubra **quantos clientes distintos** existem usando um `set`.
3. **Pipeline.** Adicione ao `csv_bruto` uma nova linha com um problema *novo* (ex.: categoria vazia). Depois acrescente uma **regra de descarte** no pipeline da seção 6 para tratá-la — e conte os descartes.
4. **Reflexão (escrever, não codar).** Pense num processo de dados do seu trabalho que hoje você faz "na mão". Descreva-o como **Extrair → Transformar → Carregar**. Traga para a próxima aula.

> Dica: o melhor exercício é **quebrar o pipeline de propósito** e consertá-lo. Erros são o melhor professor de programação.


---

## Encerramento

Este notebook acompanha o documento `.md` (teoria detalhada) e os slides `.pptx` (aula expositiva) do Encontro 1.

Hoje você: desenhou o ciclo de vida da Engenharia de Dados, viu a distância entre dado bruto e confiável, entendeu por que programamos, praticou os blocos do Python e suas estruturas, e **rodou seu primeiro pipeline ETL de ponta a ponta**.

**Bibliografia:** Reis & Housley, *Fundamentos de Engenharia de Dados* (2023) · McKinney, *Python para Análise de Dados* (2022) · Kleppmann, *Designing Data-Intensive Applications* (2017).
